In [ ]:
EXP_NAME = "SPS_ranking_test_1_17_12"
dataset_path = "synth/prev40_dim6_cont/train"
feature_map = "sps/toy_features_6"

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  from google.colab import userdata
  from google.colab import drive
  drive.mount('/content/drive')
  PROJECT_ROOT = userdata.get('PROJECT_ROOT')
else:
  PROJECT_ROOT = '../..'
  if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import json
from src.config import Config

sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.2)

sps_audit = pd.read_csv(f'{PROJECT_ROOT}/{Config.RESULTS_DIR}/{EXP_NAME}/sps_audit.csv')
sbs_audit_baseline = pd.read_csv(f'{PROJECT_ROOT}/{Config.RESULTS_DIR}/{EXP_NAME}/sps_audit_baseline.csv')
dataset = pd.read_csv(f'{PROJECT_ROOT}/{Config.DATA_DIR}/{dataset_path}.csv')

with open(f"{PROJECT_ROOT}/configs/{feature_map}.json", 'r') as f:
  features = json.load(f)

# Analysis setup

In [ ]:
full_audit = pd.concat([sps_audit, sbs_audit_baseline], ignore_index=True, axis=0)
x_desc_configs = full_audit[full_audit['bucket'] == 'x_desc'].groupby(['iteration'])['feature'].apply(set).to_dict()

iteration_per_feature = full_audit[full_audit['bucket'] == 'x_desc'].groupby('feature')[['iteration']].count()
iteration_per_feature

In [ ]:
# CHANCE LEVEL AUPRC FOR REFERENCE
positives = dataset[dataset[features["target"]["name"]] == 1]
auprc_chance = len(positives) / len(dataset)
subgroups = dataset[features["sens"][0]["name"]].unique().astype(int)

In [ ]:
# UTILITY VALUE
full_audit['auprc_value'] = full_audit['u_global_mean_auprc'] - full_audit['abla_global_mean_auprc']

# UTILITY COST
full_audit['auprc_cost'] = full_audit['x_global_mean_auprc'] - full_audit['u_global_mean_auprc']

# CF HARM REDUCTION = CF harm of the original features
full_audit['max_cf_harm_reduction'] = 0
for g in subgroups:
  full_audit['max_cf_harm_reduction'] = full_audit[[f'x_{g}_bal_harm_mean', 'max_cf_harm_reduction']].max(axis=1)

# Config trade-off analysis

In [ ]:
tradeoff_audit = full_audit.groupby('iteration')[['max_cf_harm_reduction', 'x_global_mean_auprc', 'auprc_value', 'auprc_cost']].median().round(4)

In [ ]:
costs = tradeoff_audit['auprc_cost'].values
harms = tradeoff_audit['max_cf_harm_reduction'].values
values = tradeoff_audit['auprc_value'].values

def min_max_scale(series, invert=False):
  denom = series.max() - series.min()
  if denom == 0:
    return np.ones_like(series) if invert else np.zeros_like(series)
  if invert:
    return (series.max() - series) / denom
  return (series - series.min()) / denom

tradeoff_audit['s_cost'] = min_max_scale(costs, invert=False)   # Low cost -> 0, High cost -> 1
tradeoff_audit['s_harm'] = min_max_scale(harms, invert=False)  # High fairness -> 1, Low fairness -> 0
tradeoff_audit['s_value'] = min_max_scale(values, invert=False)# High value -> 1, Low value -> 0

w = {'cost': 1, 'harm': 1, 'value': 1}

def calc_dist_to_ideal(row):
  """
  Calculate weighted squared distance to the absolute ideal point (1, 1, 1)
  """
  d2 = (
    w['cost'] * (row['s_cost'])**2 +
    w['harm'] * (row['s_harm'] - 1)**2 +
    w['value'] * (row['s_value'] - 1)**2
  )
  return round(np.sqrt(d2), 4)

tradeoff_audit['dist_to_ideal'] = tradeoff_audit.apply(calc_dist_to_ideal, axis=1)

In [ ]:
# Fairness elasticity of Utility
def calc_feu(row):
  return (row['s_cost'] / row['s_harm']).round(3) if row['s_harm'] else np.nan

tradeoff_audit['FEU'] = tradeoff_audit.apply(calc_feu, axis=1)

In [ ]:
tradeoff_audit['on_frontier'] = True

for i, current in tradeoff_audit.iterrows():
  for j, other in tradeoff_audit.iterrows():
    if (
      other['s_value'] >= current['s_value'] and 
      other['s_cost'] <= current['s_cost'] and 
      other['s_harm'] >= current['s_harm'] and 
      (
        other['s_value'] > current['s_value'] or 
        other['s_cost'] < current['s_cost'] or 
        other['s_harm'] > current['s_harm']
      )
    ):
      tradeoff_audit.loc[i, 'on_frontier'] = False
      break

pareto_frontier_3d = tradeoff_audit[tradeoff_audit['on_frontier']]
dominated_configs = tradeoff_audit[~tradeoff_audit['on_frontier']]

print("--- 3D Pareto Optimal Configurations ---")
for idx, row in pareto_frontier_3d.iterrows():
  print(f'Iteration {idx}, Xdesc: {x_desc_configs.get(idx, "empty")}')

print("\n")
print(pareto_frontier_3d.drop(['x_global_mean_auprc', 'on_frontier'], axis=1).to_markdown())


In [ ]:
best_by_feu = pareto_frontier_3d[pareto_frontier_3d['max_cf_harm_reduction'] > 0 ]
best_by_feu = pareto_frontier_3d[pareto_frontier_3d['FEU'] == pareto_frontier_3d['FEU'].min()]
print(f"Config with best Fairness Elasticity of Utility: {[x_desc_configs.get(idx, "empty") for idx in best_by_feu.index.values]}")

In [ ]:
best_by_dist_to_ideal = pareto_frontier_3d[pareto_frontier_3d['dist_to_ideal'] == pareto_frontier_3d['dist_to_ideal'].min()]
print(f"Config closest to ideal point (max value, max harm reduction, min cost): {[x_desc_configs.get(idx, "empty") for idx in best_by_dist_to_ideal.index.values]}")

## Visualisation

### 3D frontier

In [ ]:
tradeoff_3d = tradeoff_audit.reset_index(names="iteration")
tradeoff_3d['status'] = tradeoff_3d['on_frontier'].map({True: '3D Pareto Frontier', False: 'Dominated Config'})

min_auprc_val = min(tradeoff_3d['s_value'].min(), tradeoff_3d['s_cost'].min())
max_auprc_val = max(tradeoff_3d['s_value'].max(), tradeoff_3d['s_cost'].max())

axis_buffer = (max_auprc_val - min_auprc_val) * 0.05
unified_range = [min_auprc_val - axis_buffer, max_auprc_val + axis_buffer]

fig = px.scatter_3d(
    tradeoff_3d, 
    x='s_value', 
    y='s_cost', 
    z='s_harm',
    color='status',
    color_discrete_map={'3D Pareto Frontier': 'red', 'Dominated Config': 'gray'},
    hover_name='iteration',
    title='Interactive 3D VAE Trade-Off Exploration'
)

# Adjust marker sizes (make the frontier triangles pop)
fig.update_traces(marker=dict(size=6, line=dict(width=1, color='DarkSlateGrey')))

fig.update_layout(
    # Fix truncation by wiping out external margins and maximizing width/height
    width=1000,
    height=800,
    margin=dict(l=0, r=0, b=0, t=50), 
    
    # Scene configuration
    scene=dict(
        # Bind utility and cost to the exact same visual coordinate scale limits
        xaxis=dict(
            title='Scaled AUPRC Value (Maximise)', 
            range=unified_range,
            backgroundcolor="rgb(243, 243, 243)",
            gridcolor="white",
            showbackground=True,
            zerolinecolor="gray",
        ),
        yaxis=dict(
            title='Scaled AUPRC Cost (Minimise)', 
            range=unified_range,
            backgroundcolor="rgb(243, 243, 243)",
            gridcolor="white",
            showbackground=True,
            zerolinecolor="gray",
        ),
        zaxis=dict(
            title='Scaled Counterfactual Harm Reduction (Maximise)',
            backgroundcolor="rgb(230, 235, 245)",
            gridcolor="white",
            showbackground=True,
            zerolinecolor="gray",
        ),
        # Default rotation angle to see the cost vs value floor clearly
        camera=dict(
            eye=dict(x=1.6, y=1.6, z=1.2)
        )
    ),
    legend=dict(
        yanchor="top",
        y=0.95,
        xanchor="left",
        x=0.05
    )
)

fig.show()

### 2D frontiers

In [ ]:
fig = plt.figure(figsize=(20, 9))

ax1 = fig.add_subplot([0, 0, 0.4, 0.8])
ax2 = fig.add_subplot([0.45, 0, 0.4, 0.8], sharey=ax1)

# AUPRC Cost vs Max Subgroup CF harm reduction
sns.lineplot(data=pareto_frontier_3d, x='s_cost', y='s_harm', color="green", marker='', linestyle="--", errorbar=None, ax=ax1)
sns.scatterplot(data=tradeoff_audit, x='s_cost', y='s_harm', ax=ax1)
# OPTIONAL: identify the original SCM
sns.scatterplot(data=tradeoff_audit.loc[['ORIGINAL SCM']], color='#a12aa5', s=100, x='s_cost', y='s_harm', ax=ax1)

# AUPRC Value vs Max Subgroup CF harm reduction
sns.lineplot(data=pareto_frontier_3d, x='s_value', y='s_harm', color="green", marker='', linestyle="--", errorbar=None, ax=ax2)
sns.scatterplot(data=tradeoff_audit, x='s_value', y='s_harm', ax=ax2)
# OPTIONAL: identify the original SCM
sns.scatterplot(data=tradeoff_audit.loc[['ORIGINAL SCM']], color='#a12aa5', s=100, x='s_value', y='s_harm', ax=ax2)

for idx, row in pareto_frontier_3d.iterrows():
  ax1.annotate(
    str(idx),                      
    (row['s_cost'], row['s_harm']), 
    textcoords="offset points",    
    xytext=(5, 5),                 
    ha='left',
    va='bottom',  
    fontsize=9, 
    fontweight='bold', 
    color='#a12aa5'
    )
  ax2.annotate(
    str(idx),                      
    (row['s_value'], row['s_harm']), 
    textcoords="offset points",    
    xytext=(5, 5),                 
    ha='left',
    va='bottom',  
    fontsize=9, 
    fontweight='bold', 
    color='#a12aa5'
    )

ax1.set_xlim(unified_range)
ax1.xaxis.set_inverted(True)
ax2.set_xlim(unified_range)

ax1.tick_params(axis='y', which='both', right=True, labelright=False, left=False, labelleft=False)
ax1.set_xlabel('Scaled AUPRC Cost', labelpad=10)
ax1.set_ylabel('')

ax2.set_ylabel('Scaled Counterfactual Harm Reduction')
ax2.tick_params(axis='y', which='both', right=False, labelright=False, left=True, labelleft=True)
ax2.set_xlabel('Scaled AUPRC Value', labelpad=10)

ax1.legend(labels=[ 
                   'Pareto frontier', 
                   'CEVAE-HE result for each pathway configuration in the audit',
                   'Original SCM'
                   ],
           loc='upper left', bbox_to_anchor=(0, -.15), edgecolor="white", ncol=3)

plt.show()


# All configs

In [ ]:
def find_config(row):
  return x_desc_configs.get(row['iteration'], "empty Xdesc")

all_configs = tradeoff_audit.reset_index(names="iteration").drop('x_global_mean_auprc', axis=1)
all_configs['Xdesc config'] = all_configs.apply(find_config, axis=1)

In [ ]:
all_configs_by_dist_to_ideal = all_configs.sort_values(by="dist_to_ideal").reset_index(drop=True)

print(f"ORIGINAL SCM Config Rank by distance ot ideal point (out of {len(all_configs_by_dist_to_ideal)})")
print("="*50)
print(all_configs_by_dist_to_ideal[all_configs_by_dist_to_ideal['iteration'] == 'ORIGINAL SCM'].to_markdown())

print("\nAll configs:")
print("="*50)
print(all_configs_by_dist_to_ideal.to_markdown())

In [ ]:
def euclidean_dist_to_optim(row):
  return np.linalg.norm(row[['s_harm', 's_cost', 's_value']] - optimum_config_coord)

# Euclidean distance to the optimum config
optimum_config_coord = best_by_feu[['s_harm', 's_cost', 's_value']]
all_configs['optim_dist'] = all_configs.apply(euclidean_dist_to_optim, axis=1)

In [ ]:
all_configs_by_feu = all_configs.sort_values(by="FEU").reset_index(drop=True)

print(f"ORIGINAL SCM Config Rank by FEU (out of {len(all_configs_by_feu)})")
print("="*50)
print(all_configs_by_feu[all_configs_by_feu['iteration'] == 'ORIGINAL SCM'].to_markdown())

print("\nAll configs:")
print("="*50)
print(all_configs_by_feu.to_markdown())

In [ ]:
all_configs_by_dist = all_configs.sort_values(by="optim_dist").reset_index(drop=True)

print(f"ORIGINAL SCM Config Rank by Euclidean Distance to the Optimum Config (out of {len(all_configs_by_dist)})")
print("="*50)
print(all_configs_by_dist[all_configs_by_dist['iteration'] == 'ORIGINAL SCM'].to_markdown())

print("\nAll configs:")
print("="*50)
print(all_configs_by_dist.to_markdown())